# 06. Retriever

Retriever는 Vector Store를 **LangChain 체인에 연결 가능한 형태**로 감싼 인터페이스입니다.  
`as_retriever()`로 변환하면 LCEL 체인의 구성 요소로 사용할 수 있습니다.

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 벡터 스토어 준비
loader = TextLoader("data/ai_basic.txt", encoding="utf-8")
docs = loader.load()
chunks = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50).split_documents(docs)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

print("준비 완료")

준비 완료


## 1. 기본 Retriever 생성

In [2]:
# k: 반환할 문서 수
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

results = retriever.invoke("RAG란 무엇인가요?")

print(f"검색된 문서 수: {len(results)}")
for i, doc in enumerate(results):
    print(f"\n[{i+1}] {doc.page_content[:150]}")

검색된 문서 수: 3

[1] RAG(Retrieval-Augmented Generation)이란?
RAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 위해 외부 문서를 검색하여 그 내용을 바탕으로 답변을 생성합니다. 최신 정보 제공, 할루시

[2] LangChain이란?
LangChain은 LLM 기반 애플리케이션 개발을 위한 오픈소스 프레임워크입니다. 모델, 프롬프트, 파서, 체인, 에이전트, 툴 등 다양한 컴포넌트를 제공합니다. Python과 JavaScript 버전이 있으며, RAG, 에이전트, 챗봇 등 다

[3] 트랜스포머(Transformer)란?
트랜스포머는 2017년 구글이 발표한 신경망 구조로, 자연어 처리 분야에 혁신을 가져왔습니다. 어텐션 메커니즘(Attention Mechanism)을 핵심으로 사용하며, 병렬 처리가 가능해 학습 속도가 빠릅니다. BERT, GPT 


## 2. 검색 타입 설정

| search_type | 설명 |
|---|---|
| `similarity` | 코사인 유사도 기반 (기본값) |
| `mmr` | 다양성을 고려한 검색 (중복 최소화) |
| `similarity_score_threshold` | 유사도 임계값 이상만 반환 |

In [3]:
# MMR - 유사하면서도 다양한 문서 검색
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10}
)

results = mmr_retriever.invoke("머신러닝에 대해 알려주세요")
for i, doc in enumerate(results):
    print(f"[{i+1}] {doc.page_content[:150]}")
    print()

[1] 인공지능(AI)의 기초

인공지능이란 무엇인가?
인공지능(Artificial Intelligence, AI)은 컴퓨터 시스템이 인간의 지능적 행동을 모방하도록 하는 기술입니다. 학습, 추론, 문제 해결, 언어 이해 등의 능력을 포함합니다.

머신러닝이란?
머신러닝(Ma

[2] 벡터 스토어(Vector Store)란?
벡터 스토어는 임베딩 벡터를 저장하고 유사도 검색을 수행하는 데이터베이스입니다. FAISS, Chroma, Pinecone, Weaviate 등이 대표적입니다. 코사인 유사도나 내적을 사용해 쿼리와 가장 유사한 문서를 빠르게 찾

[3] 합성곱 신경망(CNN, Convolutional Neural Network)은 이미지 처리에 특화된 신경망 구조입니다. 필터를 사용해 이미지의 특징을 추출하며, 이미지 분류, 객체 탐지에 사용됩니다.

순환 신경망(RNN, Recurrent Neural Network)



In [4]:
# score_threshold - 유사도 임계값 설정
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.5, "k": 5}
)

results = threshold_retriever.invoke("딥러닝이란?")
print(f"임계값 이상 문서 수: {len(results)}")
for doc in results:
    print("-", doc.page_content[:100])

e:\langgraph_modular_rag\.venv\Lib\site-packages\langchain_core\vectorstores\base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='c2a41992-f756-4011-b5e3-dbdc4d38b33b', metadata={'source': 'data/ai_basic.txt'}, page_content='강화학습(Reinforcement Learning)은 에이전트가 환경과 상호작용하면서 보상을 최대화하는 방향으로 학습합니다. 게임 AI, 로봇 제어 등에 활용됩니다.\n\n딥러닝이란?\n딥러닝(Deep Learning)은 여러 층의 인공 신경망을 사용하는 머신러닝 기법입니다. 이미지 인식, 자연어 처리, 음성 인식 등에서 뛰어난 성능을 보입니다.'), np.float32(0.3020758)), (Document(id='47d55485-b84e-4049-9cda-aef9ca18c754', metadata={'source': 'data/ai_basic.txt'}, page_content='합성곱 신경망(CNN, Convolutional Neural Network)은 이미지 처리에 특화된 신경망 구조입니다. 필터를 사용해 이미지의 특징을 추출하며, 이미지 분류, 객체 탐지에 사용됩니다.\n\n순환 신경망(RNN, Recurrent Neural Network)은 순서가 있는 데이터(시계열, 텍스트)를 처리합니다. LSTM과 GRU는 RNN의 장기 의존성 문제를 해결한 개선된 구조입니다.'), np.float32(-0.049161434)), (Document(id='295903c1-74a5-4c85-8a19-d7a7b103aa29', metadata={'source': 'data/ai_basic.txt'}, page_content='지도학습(Supervised Learning)은 입력과 정답 레이블이 쌍으로 주어

임계값 이상 문서 수: 0


## 3. Retriever를 LCEL 체인에 연결

In [5]:
from langchain_core.runnables import RunnableLambda

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 검색 결과를 하나의 문자열로 합치는 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# retriever → format 체인
retrieve_chain = retriever | RunnableLambda(format_docs)

context = retrieve_chain.invoke("임베딩이란?")
print(context)

임베딩(Embedding)이란?
임베딩은 텍스트를 숫자 벡터로 변환하는 기술입니다. 의미가 비슷한 텍스트는 벡터 공간에서 가까운 위치에 놓입니다. OpenAI의 text-embedding-ada-002, text-embedding-3-small 등이 널리 사용됩니다.

벡터 스토어(Vector Store)란?
벡터 스토어는 임베딩 벡터를 저장하고 유사도 검색을 수행하는 데이터베이스입니다. FAISS, Chroma, Pinecone, Weaviate 등이 대표적입니다. 코사인 유사도나 내적을 사용해 쿼리와 가장 유사한 문서를 빠르게 찾습니다.

합성곱 신경망(CNN, Convolutional Neural Network)은 이미지 처리에 특화된 신경망 구조입니다. 필터를 사용해 이미지의 특징을 추출하며, 이미지 분류, 객체 탐지에 사용됩니다.

순환 신경망(RNN, Recurrent Neural Network)은 순서가 있는 데이터(시계열, 텍스트)를 처리합니다. LSTM과 GRU는 RNN의 장기 의존성 문제를 해결한 개선된 구조입니다.


## 4. MultiQueryRetriever 맛보기

- 단일 쿼리를 여러 쿼리로 확장해 더 풍부한 검색 결과를 가져옴(Advanced RAG에서 자세히 다룹니다)  
- 하나의 입력 질문에 LLM이 여러 관점의 검색 질문을 바꿔서 검색하고 결과를 가져옴

- langchain.retrievers 모듈을 쓰려면 langchain-classic을 설치해야함.  
    ```uv add langchain-classic```

In [7]:
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
base_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

multi_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)

results = multi_retriever.invoke("AI의 학습 방법을 알려줘")
print(f"검색된 문서 수: {len(results)}")
for doc in results[:3]:
    print("-", doc.page_content[:120])

검색된 문서 수: 3
- 인공지능(AI)의 기초

인공지능이란 무엇인가?
인공지능(Artificial Intelligence, AI)은 컴퓨터 시스템이 인간의 지능적 행동을 모방하도록 하는 기술입니다. 학습, 추론, 문제 해결, 언어 이해 
- 강화학습(Reinforcement Learning)은 에이전트가 환경과 상호작용하면서 보상을 최대화하는 방향으로 학습합니다. 게임 AI, 로봇 제어 등에 활용됩니다.

딥러닝이란?
딥러닝(Deep Learning)은
- 지도학습(Supervised Learning)은 입력과 정답 레이블이 쌍으로 주어진 데이터를 학습합니다. 분류(Classification)와 회귀(Regression) 문제에 사용됩니다. 대표적인 알고리즘으로는 선형


## 정리

- `as_retriever()`로 VectorStore를 LCEL 체인에 연결합니다.
- `similarity`, `mmr`, `similarity_score_threshold` 검색 방식 선택 가능
- Retriever는 `invoke(query)` → `List[Document]` 반환

다음 단계: **전체 RAG 파이프라인 구현** →